In [1]:
# In a Kaggle cell, paste the entire notebook as a Python script instead:
!pip install -q pot scikit-learn matplotlib pandas

In [2]:
import numpy as np
import json
import time
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
import ot
print("OK")

2026-05-06 12:40:27.984162: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778071228.183050      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778071228.241630      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778071228.763546      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778071228.763590      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778071228.763593      22 computation_placer.cc:177] computation placer alr

OK


In [3]:
N_FIELD = 2000
K = 32
SEEDS = [0, 1, 2]
time_grid = np.linspace(0.05, 0.95, 19)

def cond_var_knn(Xt, ut, k=32):
    nn = NearestNeighbors(n_neighbors=k).fit(Xt)
    _, idx = nn.kneighbors(Xt)
    n = len(ut)
    tv = 0.0
    for i in range(n):
        nb = ut[idx[i]]
        m = nb.mean(0)
        tv += np.sum((nb - m)**2) / k
    return tv / n

obs_gaps, aug_gaps = [], []
for seed in SEEDS:
    rng = np.random.RandomState(seed)
    S1 = rng.uniform(0, 1, N_FIELD)
    S2 = rng.uniform(0, 1, N_FIELD)
    U_pos = np.ones((N_FIELD, 1))
    U_neg = -np.ones((N_FIELD, 1))
    H_pos = np.ones((N_FIELD, 1))
    H_neg = -np.ones((N_FIELD, 1))
    
    gc_obs, gc_aug = [], []
    for t in time_grid:
        Xt = np.concatenate([(S1+t).reshape(-1,1), (S2+1-t).reshape(-1,1)])
        Ut = np.concatenate([U_pos, U_neg])
        Ht = np.concatenate([H_pos, H_neg])
        Xt_aug = np.concatenate([Xt, Ht], axis=1)
        gc_obs.append(cond_var_knn(Xt, Ut, K))
        gc_aug.append(cond_var_knn(Xt_aug, Ut, K))
    
    obs_gaps.append(np.trapz(gc_obs, time_grid))
    aug_gaps.append(np.trapz(gc_aug, time_grid))
    print(f"seed={seed}: observed={obs_gaps[-1]:.4f}, augmented={aug_gaps[-1]:.6f}")

print(f"\nObserved: {np.mean(obs_gaps):.4f} +/- {np.std(obs_gaps):.4f}")
print(f"Augmented: {np.mean(aug_gaps):.6f} +/- {np.std(aug_gaps):.6f}")

/tmp/ipykernel_22/1087674376.py:36: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  obs_gaps.append(np.trapz(gc_obs, time_grid))
/tmp/ipykernel_22/1087674376.py:37: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  aug_gaps.append(np.trapz(gc_aug, time_grid))


seed=0: observed=0.4837, augmented=0.000000


/tmp/ipykernel_22/1087674376.py:36: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  obs_gaps.append(np.trapz(gc_obs, time_grid))
/tmp/ipykernel_22/1087674376.py:37: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  aug_gaps.append(np.trapz(gc_aug, time_grid))


seed=1: observed=0.4806, augmented=0.000000
seed=2: observed=0.4831, augmented=0.000000

Observed: 0.4825 +/- 0.0013
Augmented: 0.000000 +/- 0.000000


/tmp/ipykernel_22/1087674376.py:36: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  obs_gaps.append(np.trapz(gc_obs, time_grid))
/tmp/ipykernel_22/1087674376.py:37: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  aug_gaps.append(np.trapz(gc_aug, time_grid))


In [4]:
om, os = np.mean(obs_gaps), np.std(obs_gaps)
am, astd = np.mean(aug_gaps), np.std(aug_gaps)
print(rf"""
\begin{{table}}[t]
\centering\small
\caption{{Field-line crossing diagnostic.}}
\label{{tab:fieldline_crossing}}
\begin{{tabular}}{{@{{}}lcc@{{}}}}
\toprule
Conditioning state & Estimated gap $\downarrow$ & Interpretation \\
\midrule
Observed $X_t$ & ${om:.3f} \pm {os:.3f}$ & branch info lost \\
Augmented $(X_t, H)$ & ${am:.5f} \pm {astd:.5f}$ & branch info retained \\
\bottomrule
\end{{tabular}}
\end{{table}}
""")


\begin{table}[t]
\centering\small
\caption{Field-line crossing diagnostic.}
\label{tab:fieldline_crossing}
\begin{tabular}{@{}lcc@{}}
\toprule
Conditioning state & Estimated gap $\downarrow$ & Interpretation \\
\midrule
Observed $X_t$ & $0.482 \pm 0.001$ & branch info lost \\
Augmented $(X_t, H)$ & $0.00000 \pm 0.00000$ & branch info retained \\
\bottomrule
\end{tabular}
\end{table}



In [5]:
N = 1024
cifar = torchvision.datasets.CIFAR10("./data", train=True, download=True,
    transform=T.Compose([T.ToTensor(), T.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))]))
cifar_flat = torch.stack([cifar[i][0] for i in range(N*4)]).reshape(N*4, -1).numpy()

fmnist = torchvision.datasets.FashionMNIST("./data", train=True, download=True,
    transform=T.Compose([T.ToTensor(), T.Normalize((0.5,),(0.5,))]))
fmnist_flat = torch.stack([fmnist[i][0] for i in range(N*4)]).reshape(N*4, -1).numpy()
print(f"CIFAR: {cifar_flat.shape}, FMNIST: {fmnist_flat.shape}")

100%|██████████| 170M/170M [05:38<00:00, 504kB/s]
100%|██████████| 26.4M/26.4M [00:00<00:00, 117MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 4.16MB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 57.9MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 2.45MB/s]


CIFAR: (4096, 3072), FMNIST: (4096, 784)


In [6]:
def ot_coupling(Z, X, bs=128):
    n = len(Z)
    perm = np.arange(n)
    for s in range(0, n, bs):
        e = min(s+bs, n)
        C = np.sum((Z[s:e,None,:]-X[None,s:e,:])**2, 2)
        P = ot.emd(np.ones(e-s)/(e-s), np.ones(e-s)/(e-s), C)
        perm[s:e] = s + np.argmax(P, 1)
    return perm

results = []
for dname, data in [("CIFAR-10", cifar_flat), ("Fashion-MNIST", fmnist_flat)]:
    for coupling in ["independent", "minibatch_ot"]:
        raw_list, eta_list = [], []
        for seed in [0,1,2]:
            rng = np.random.RandomState(seed)
            idx = rng.choice(len(data), N, replace=False)
            X = data[idx]
            Z = rng.standard_normal(X.shape).astype(np.float32)
            if coupling == "minibatch_ot":
                if X.shape[1] > 784:
                    Zd = F.avg_pool2d(torch.from_numpy(Z.reshape(-1,3,32,32)).float(),8).reshape(N,-1).numpy()
                    Xd = F.avg_pool2d(torch.from_numpy(X.reshape(-1,3,32,32)).float(),8).reshape(N,-1).numpy()
                else:
                    Zd, Xd = Z, X
                X = X[ot_coupling(Zd, Xd)]
            
            comb = np.concatenate([Z, X])
            dim = min(64, comb.shape[1])
            pca = PCA(dim).fit(comb)
            U = X - Z
            Uf = pca.transform(U)
            mvar = np.trace(np.cov(Uf.T))
            
            cv_curve = []
            for t in time_grid:
                Xt = (1-t)*Z + t*X
                Xtf = pca.transform(Xt)
                cv_curve.append(cond_var_knn(Xtf, Uf, K))
            
            raw = np.trapz(cv_curve, time_grid)
            eta = raw / (mvar * (0.95-0.05)) if mvar > 0 else float("nan")
            raw_list.append(raw)
            eta_list.append(eta)
            print(f"  seed={seed} {dname} {coupling}: raw={raw:.3f} eta={eta:.4f}")
        
        results.append({"dataset": dname, "coupling": coupling,
                        "raw_mean": np.mean(raw_list), "raw_std": np.std(raw_list),
                        "eta_mean": np.mean(eta_list), "eta_std": np.std(eta_list)})
        print(f"  >>> {dname} {coupling}: eta={np.mean(eta_list):.4f}\n")

/tmp/ipykernel_22/3942801800.py:41: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  raw = np.trapz(cv_curve, time_grid)


  seed=0 CIFAR-10 independent: raw=483.777 eta=0.6056


/tmp/ipykernel_22/3942801800.py:41: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  raw = np.trapz(cv_curve, time_grid)


  seed=1 CIFAR-10 independent: raw=483.835 eta=0.6014


/tmp/ipykernel_22/3942801800.py:41: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  raw = np.trapz(cv_curve, time_grid)


...

/tmp/ipykernel_22/3942801800.py:41: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  raw = np.trapz(cv_curve, time_grid)


  seed=0 Fashion-MNIST minibatch_ot: raw=175.050 eta=0.5971


/tmp/ipykernel_22/3942801800.py:41: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  raw = np.trapz(cv_curve, time_grid)


  seed=1 Fashion-MNIST minibatch_ot: raw=172.785 eta=0.5990
  seed=2 Fashion-MNIST minibatch_ot: raw=172.552 eta=0.5938
  >>> Fashion-MNIST minibatch_ot: eta=0.5966



/tmp/ipykernel_22/3942801800.py:41: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  raw = np.trapz(cv_curve, time_grid)


In [7]:
print(r"""\begin{table}[t]
\centering\small
\caption{Normalized proxy-gap control. $\eta$ = unexplained fraction of velocity variance.}
\label{tab:normalized_gap}
\begin{tabular}{@{}llcc@{}}
\toprule
Dataset & Coupling & Proxy gap $\downarrow$ & Normalized $\eta\downarrow$ \\
\midrule""")
for r in results:
    cn = "Independent" if r["coupling"]=="independent" else "Minibatch OT"
    print(f"{r['dataset']} & {cn} & ${r['raw_mean']:.2f} \\pm {r['raw_std']:.2f}$ & ${r['eta_mean']:.4f} \\pm {r['eta_std']:.4f}$ \\\\")
print(r"""\bottomrule
\end{tabular}
\end{table}""")

\begin{table}[t]
\centering\small
\caption{Normalized proxy-gap control. $\eta$ = unexplained fraction of velocity variance.}
\label{tab:normalized_gap}
\begin{tabular}{@{}llcc@{}}
\toprule
Dataset & Coupling & Proxy gap $\downarrow$ & Normalized $\eta\downarrow$ \\
\midrule
CIFAR-10 & Independent & $484.89 \pm 1.53$ & $0.6022 \pm 0.0025$ \\
CIFAR-10 & Minibatch OT & $430.84 \pm 3.30$ & $0.5962 \pm 0.0075$ \\
Fashion-MNIST & Independent & $209.80 \pm 1.39$ & $0.6148 \pm 0.0018$ \\
Fashion-MNIST & Minibatch OT & $173.46 \pm 1.13$ & $0.5966 \pm 0.0022$ \\
\bottomrule
\end{tabular}
\end{table}
